# Часть A. Подготовка: машина и окружения

Готовим всё для работы: Google Drive, токен Hugging Face, проверку машины и два окружения Python — `unl` (обучение и оценка моделей кодом OpenUnlearning) и `atk` (атаки и анализ: vLLM, spaCy, BERTScore, пакет `urec`). Раньше это были две части плана, A и B.

Перед запуском выберите GPU: Runtime → Change runtime type → L4, A100 или H100 (T4 для обучения не подходит). Всё вместе — около 15–20 минут.

Блокнот только рассказывает, что происходит, и вызывает функции. Сами команды (git, uv, nvidia-smi) с подробными комментариями — в файле [`scripts/colab_setup.py`](https://github.com/IvanovskyDev/Machine-Unlearning-Diploma/blob/main/scripts/colab_setup.py).

**0. Код проекта.** Клонируем репозиторий диплома вместе с форком OpenUnlearning и подключаем модуль `colab_setup`. Такая ячейка нужна в начале каждого блокнота: без неё функций `colab_setup` ещё нет. Чтобы проверить Pull Request до слияния, впишите в `BRANCH` его ветку.

In [ ]:
import os
import subprocess
import sys

REPO = "/content/repo"   # сюда клонируется репозиторий диплома
URL = "https://github.com/IvanovskyDev/Machine-Unlearning-Diploma.git"
BRANCH = "main"          # какую ветку взять; для проверки Pull Request — его ветку

# --recurse-submodules — вместе с репозиторием скачать форк OpenUnlearning (он подключён как submodule);
# если репозиторий уже скачан в этой сессии, второй раз не качаем
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--recurse-submodules", "-b", BRANCH, URL, REPO], check=True)

sys.path.append(REPO + "/scripts")   # чтобы import нашёл scripts/colab_setup.py
import colab_setup

**1. Старт сессии**: подключаем Google Drive (Colab попросит разрешение — согласитесь), создаём папки, связываем `saves` форка и `results/raw` с Drive, задаём переменные окружения и берём токен Hugging Face из секрета `HF_TOKEN`. Должны напечататься ваше имя на Hugging Face и коммиты репозитория и форка.

Токен добавляется заранее, один раз: huggingface.co → Settings → Access Tokens → создать токен типа Read; в Colab слева 🔑 Secrets → добавить `HF_TOKEN` и включить доступ для блокнота.

In [ ]:
colab_setup.start_session()

**2. Смотрим машину.** GPU должна быть L4, A100 или H100, а CUDA Version — 12 или выше.

In [ ]:
print(colab_setup.machine_info())

**3. Собираем окружения `unl` и `atk`** ровно по lock-файлам из `envs/` репозитория, затем ставим FlashAttention в `unl` и наш пакет `urec` в `atk`. Займёт несколько минут; в конце должно быть `All installed packages are compatible`.

In [ ]:
colab_setup.build_envs()

**4. Самопроверка окружений**: пакеты совпадают с lock-файлами, GPU видна, FlashAttention, vLLM, spaCy и BERTScore работают, тесты `urec` проходят. По каждой проверке печатается OK или FAIL, в конце — `OK: все проверки прошли`. Около трёх минут: в первый раз BERTScore скачивает модель roberta-large (1,4 ГБ).

In [ ]:
colab_setup.check_envs()

**5. Скачиваем оценки готовых моделей от авторов OpenUnlearning** — по ним считается Forget Quality. Файлы попадают в `saves/eval` на Drive, поэтому этот шаг нужен один раз на весь проект. В конце печатается список папок `tofu_…`.

In [ ]:
colab_setup.download_eval_logs()

**6. Записываем в журнал** (`journal.md` на Drive, пригодится для главы 3 диплома): характеристики машины — один раз для каждой новой модели GPU, окружения — после каждой пересборки lock-файлов.

In [ ]:
colab_setup.write_journal("Проверка машины (Colab)", colab_setup.machine_info())
colab_setup.write_journal("Окружения unl и atk (Colab)", colab_setup.envs_info())

**7. Завершаем сессию**: дожидаемся, пока все файлы запишутся на Drive, и отключаем его. Чтобы продолжить работу после этого шага, начните снова с шага 0.

In [ ]:
colab_setup.finish_session()

**Приложение. Как обновить версии пакетов.** Окружения собираются только по lock-файлам, поэтому версии сами не меняются. Чтобы их сменить, поправьте списки `UNL_PACKAGES` и `ATK_PACKAGES` в `scripts/colab_setup.py` и в новой сессии выполните шаги 0–1 и `colab_setup.build_envs_from_scratch()`. Новые lock-файлы появятся в `envs/` на Drive: скопируйте их в `envs/` репозитория, закоммитьте и прогоните блокноты заново.